# 02. Silao Geo-Fenced OSINT Incident Pipeline

**Objective:** Ingest, classify (ES/EN), geocode via offline Silao gazetteer, deduplicate, and persist external local-disruption events (`traffic_accident`, `road_closure`, `public_safety`, `weather_disruption`) for downstream correlation and ROI modeling in Notebooks `03`–`05`.

**Mode Contract (`sample` vs. `raw`):**
- `BGA_DATA_MODE="sample"`: uses `tests/fixtures/sample_feed.xml` + `data/sample/incidents.csv` (plus opt-in live Google News RSS if `BGA_FETCH_LIVE=1`).
- `BGA_DATA_MODE="raw"`: **strictly excludes** sample/test fixtures; discovers local raw files in `data/raw/` and/or fetches live RSS when `BGA_FETCH_LIVE=1`.
- **Handoff Artifact:** Persists deduplicated events to `data/processed/incidents_<DATA_MODE>.csv`.

## 1. Environment Setup & OSINT Imports

**Objective:** Configure PID-isolated matplotlib environment, inline rendering helpers, `StoreConfig`, and `FETCH_LIVE` flag.

**Inputs / Assumptions:** `BGA_FETCH_LIVE="0"` by default so offline runs are deterministic.

**How to Read the Output:** Displays resolved `DATA_MODE`, `FETCH_LIVE`, and the opt-in Google News RSS endpoint URL.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("BGA_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config
from src.notebook_utils import (
    configure_mpl_env,
    processed_incidents_path,
    render_table,
    show_fig,
)
from src.osint_collector import (
    STANDARD_INCIDENT_COLUMNS,
    build_google_news_rss_url,
    build_incident_dataset,
)

plt = configure_mpl_env()
DATA_MODE = os.environ.get("BGA_DATA_MODE", "sample").strip().lower()
FETCH_LIVE = os.environ.get("BGA_FETCH_LIVE", "0") == "1"
cfg = load_config(repo_root=REPO_ROOT)
print(f"DATA_MODE={DATA_MODE} | FETCH_LIVE={FETCH_LIVE} | RSS={build_google_news_rss_url()}")

## 2. Mode-Isolated OSINT Ingestion & Deduplication

**Objective:** Collect and deduplicate OSINT events strictly from mode-appropriate sources with explicit exception handling for live RSS failures.

**Inputs / Assumptions:** `sample` mode combines fixture XML and sample CSV; `raw` mode scans `data/raw/*incident*.{csv,json,xml}` plus opt-in RSS.

**How to Read the Output:** Inspect ingestion KPI summary (count, date span, geocode coverage) and top-10 normalized incident rows.

In [ ]:
xml_texts = []
file_paths = []
if DATA_MODE == "sample":
    fixture_path = REPO_ROOT / "tests" / "fixtures" / "sample_feed.xml"
    if fixture_path.is_file():
        xml_texts.append(fixture_path.read_text(encoding="utf-8"))
    sample_csv = cfg.sample_dir / "incidents.csv"
    if sample_csv.is_file():
        file_paths.append(sample_csv)
elif DATA_MODE == "raw":
    if cfg.raw_dir.is_dir():
        for fp in sorted(cfg.raw_dir.glob("*incident*.xml")):
            xml_texts.append(fp.read_text(encoding="utf-8"))
        for fp in sorted(list(cfg.raw_dir.glob("*incident*.csv")) + list(cfg.raw_dir.glob("*incident*.json"))):
            file_paths.append(fp)
else:
    raise ValueError(f"Unsupported DATA_MODE={DATA_MODE!r}")

feed_urls = [build_google_news_rss_url()] if FETCH_LIVE else ()
if not xml_texts and not file_paths and not feed_urls:
    print(f"WARNING: No local *incident* files in {cfg.raw_dir} and FETCH_LIVE=False; producing empty incident dataset.")
    incidents_df = pd.DataFrame(columns=STANDARD_INCIDENT_COLUMNS)
else:
    try:
        incidents_df = build_incident_dataset(
            xml_texts=xml_texts,
            file_paths=file_paths,
            feed_urls=feed_urls,
            use_nominatim=False,
            relevant_only=True,
        )
    except Exception as exc:
        print(f"WARNING: Live RSS fetch failed ({exc}); retrying offline local sources only.")
        incidents_df = (
            build_incident_dataset(xml_texts=xml_texts, file_paths=file_paths, feed_urls=(), use_nominatim=False, relevant_only=True)
            if (xml_texts or file_paths)
            else pd.DataFrame(columns=STANDARD_INCIDENT_COLUMNS)
        )

date_span = (
    f"{incidents_df['date'].min()} .. {incidents_df['date'].max()}"
    if not incidents_df.empty
    else "N/A"
)
kpi_tbl = pd.DataFrame([
    {
        "mode": DATA_MODE,
        "total_incidents": len(incidents_df),
        "date_span": date_span,
        "categories": int(incidents_df["category"].nunique()) if not incidents_df.empty else 0,
        "geocode_methods": ", ".join(f"{k}:{v}" for k, v in incidents_df["geocode_method"].value_counts().items()) if not incidents_df.empty else "none",
    }
])
render_table(kpi_tbl, title="OSINT Ingestion Summary")
if not incidents_df.empty:
    render_table(
        incidents_df[["date", "hour_int", "category", "severity", "location_name", "geocode_method"]].head(10),
        title="Top 10 Deduplicated OSINT Incidents",
        fmt={"severity": "{:.2f}"},
    )
else:
    print("WARNING: Zero relevant OSINT incidents collected.")

## 3. Persist Downstream OSINT Artifact

**Objective:** Write normalized `incidents_df` to `data/processed/incidents_<DATA_MODE>.csv` for deterministic handoff to Notebooks `03`, `04`, and `05`.

**Inputs / Assumptions:** `data/processed/` is gitignored; empty runs still emit a valid schema header.

**How to Read the Output:** Confirms output path, row count, and file size.

In [ ]:
PROCESSED_INCIDENTS_PATH = processed_incidents_path(cfg, DATA_MODE)
PROCESSED_INCIDENTS_PATH.parent.mkdir(parents=True, exist_ok=True)
out_df = incidents_df if not incidents_df.empty else pd.DataFrame(columns=STANDARD_INCIDENT_COLUMNS)
out_df.to_csv(PROCESSED_INCIDENTS_PATH, index=False)
print(f"Wrote {len(out_df)} incidents -> {PROCESSED_INCIDENTS_PATH} ({PROCESSED_INCIDENTS_PATH.stat().st_size} bytes)")

## 4. Category & Hourly Disruption Distribution

**Objective:** Inspect incident frequency, mean severity by category, and 24-hour arrival alignment against storefront operating hours.

**Inputs / Assumptions:** Operating hours shaded (`cfg.operating_hours`); skips plot gracefully if dataset is empty.

**How to Read the Output:** Identifies dominant shock categories and whether events align with store operating/peak hours.

In [ ]:
if incidents_df.empty:
    print("Skipping OSINT plots: zero incidents available.")
else:
    cat_stats = incidents_df.groupby("category", as_index=False).agg(
        count=("incident_id", "count"),
        mean_severity=("severity", "mean"),
    )
    render_table(cat_stats, title="Incidents by Bilingual Category", fmt={"mean_severity": "{:.2f}"})

    hr_counts = (
        pd.to_numeric(incidents_df["hour_int"], errors="coerce")
        .dropna()
        .astype(int)
        .value_counts()
        .reindex(range(24), fill_value=0)
    )
    op_start, op_end = cfg.operating_hours

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
    bars = axes[0].bar(cat_stats["category"], cat_stats["count"], color="#2b8cbe")
    for bar, sev in zip(bars, cat_stats["mean_severity"]):
        axes[0].text(bar.get_x() + bar.get_width() / 2.0, bar.get_height() + 0.15, f"sev={sev:.2f}", ha="center", va="bottom", fontsize=8.5)
    axes[0].set_title("OSINT Incidents by Category (with Mean Severity)")
    axes[0].set_ylabel("Incident Count")
    axes[0].tick_params(axis="x", rotation=15)
    axes[0].set_ylim(0, max(cat_stats["count"]) * 1.2 + 0.5)

    axes[1].axvspan(op_start - 0.5, op_end + 0.5, color="#fef0d9", alpha=0.8, label=f"Operating ({op_start:02d}–{op_end:02d})")
    axes[1].bar(hr_counts.index, hr_counts.values, color="#e34a33", alpha=0.85, label="Incidents")
    axes[1].set_xticks(range(0, 24, 3))
    axes[1].set_title("24-Hour Incident Distribution vs Operating Window")
    axes[1].set_xlabel("Hour of Day")
    axes[1].set_ylabel("Incident Count")
    axes[1].legend()
    show_fig(fig)